In [0]:
# ==========================================================
# 06_silver_alunos
# Camada: Silver
# Fonte: 2026-11-01_alunos.parquet (Bronze)
# Destino: default.alunos
#
# RESPONSABILIDADE
# Transformar a base de alunos da avaliação de alfabetização
# em tabela Silver confiável, com tipagem e DQ.
#
# NOTA DE VOLUME
# 3,87 milhões de registros. As agregações da célula 5 são
# feitas em uma única passada para evitar múltiplas varreduras.
#
# NOTA SOBRE NULOS
# proficiencia e peso_aluno são nulos para alunos ausentes
# (presenca = 0). Isso é dado válido, não falha de qualidade.
#
# NOTA DE PERMISSÃO
# O schema silver pertence a outro usuário. A gravação ocorre
# em default, conforme adaptação documentada na seção 10.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "alunos"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_alunos.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - presenca, preenchimento_caderno e alfabetizado são
#   convertidos de string para int.
# - proficiencia e peso_aluno permanecem double e podem ser
#   nulos (alunos ausentes).
# - Dedupe por (ano, id_municipio, id_escola, id_aluno).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("id_municipio").cast("string").alias("id_municipio"),
        F.col("id_escola").cast("string").alias("id_escola"),
        F.col("id_aluno").cast("string").alias("id_aluno"),
        F.col("caderno").cast("string").alias("caderno"),
        F.col("serie").cast("string").alias("serie"),
        F.col("rede").cast("string").alias("rede"),
        F.col("presenca").cast("int").alias("presenca"),
        F.col("preenchimento_caderno").cast("int").alias("preenchimento_caderno"),
        F.col("alfabetizado").cast("int").alias("alfabetizado"),
        F.col("proficiencia").cast("double").alias("proficiencia"),
        F.col("peso_aluno").cast("double").alias("peso_aluno"),
    )
    .filter(F.col("ano").isNotNull() & F.col("id_aluno").isNotNull())
    .dropDuplicates(["ano", "id_municipio", "id_escola", "id_aluno"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.select(
    "ano", "id_municipio", "id_escola", "id_aluno",
    "rede", "presenca", "alfabetizado", "proficiencia",
    "ingested_at", "source", "version"
).show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Agregação em uma única passada para eficiência em 3,87M.
#
# ATENÇÃO: proficiencia nula NÃO é falha crítica. Alunos
# ausentes (presenca = 0) não têm proficiência calculada.
# ==========================================================

metricas = df_silver.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("id_municipio").isNull(), 1).otherwise(0)).alias("nulos_municipio"),
    F.sum(F.when(F.col("id_escola").isNull(), 1).otherwise(0)).alias("nulos_escola"),
    F.sum(F.when(F.col("id_aluno").isNull(), 1).otherwise(0)).alias("nulos_aluno"),
    F.sum(F.when(F.col("rede").isNull(), 1).otherwise(0)).alias("nulos_rede"),
    F.sum(F.when(F.col("presenca").isNull(), 1).otherwise(0)).alias("nulos_presenca"),
    F.sum(F.when(F.col("alfabetizado").isNull(), 1).otherwise(0)).alias("nulos_alfabetizado"),
    F.sum(F.when(F.col("proficiencia").isNull(), 1).otherwise(0)).alias("nulos_proficiencia"),
    F.sum(F.when(~F.col("presenca").isin(0, 1), 1).otherwise(0)).alias("presenca_invalida"),
    F.sum(F.when(~F.col("alfabetizado").isin(0, 1), 1).otherwise(0)).alias("alfabetizado_invalido"),
    F.sum(F.when((F.col("proficiencia") < 0) | (F.col("proficiencia") > 1000), 1).otherwise(0)).alias("proficiencia_fora_faixa"),
    F.countDistinct("id_municipio").alias("municipios"),
    F.countDistinct("id_escola").alias("escolas"),
    F.countDistinct("id_aluno").alias("alunos"),
).collect()[0]

total = metricas["total"]
chave = ["ano", "id_municipio", "id_escola", "id_aluno"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

print("=" * 55)
print("DATA QUALITY — alunos")
print("=" * 55)
print(f"Registros totais       : {total:,}")
print(f"Município nulo         : {metricas['nulos_municipio']:,}")
print(f"Escola nula            : {metricas['nulos_escola']:,}")
print(f"Aluno nulo             : {metricas['nulos_aluno']:,}")
print(f"Rede nula              : {metricas['nulos_rede']:,}")
print(f"Presença nula          : {metricas['nulos_presenca']:,}")
print(f"Alfabetizado nulo      : {metricas['nulos_alfabetizado']:,}")
print(f"Proficiência nula      : {metricas['nulos_proficiencia']:,} ({metricas['nulos_proficiencia']/total*100:.2f}%)")
print(f"Presença fora de 0/1   : {metricas['presenca_invalida']:,}")
print(f"Alfabetizado fora 0/1  : {metricas['alfabetizado_invalido']:,}")
print(f"Profic. fora de 0-1000 : {metricas['proficiencia_fora_faixa']:,}")
print(f"Duplicatas na chave    : {duplicatas:,}")
print(f"Municípios distintos   : {metricas['municipios']:,}")
print(f"Escolas distintas      : {metricas['escolas']:,}")
print(f"Alunos distintos       : {metricas['alunos']:,}")
print("=" * 55)

assert metricas["nulos_municipio"] == 0, "FALHA CRÍTICA: id_municipio nulo"
assert metricas["nulos_escola"] == 0, "FALHA CRÍTICA: id_escola nulo"
assert metricas["nulos_aluno"] == 0, "FALHA CRÍTICA: id_aluno nulo"
assert metricas["presenca_invalida"] == 0, "FALHA CRÍTICA: presenca fora de 0/1"
assert metricas["alfabetizado_invalido"] == 0, "FALHA CRÍTICA: alfabetizado fora de 0/1"
assert metricas["proficiencia_fora_faixa"] == 0, "FALHA CRÍTICA: proficiência fora de 0-1000"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# Particionado por ano para otimizar leitura analítica.
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")

print("\nCobertura por ano:")
df_check.groupBy("ano").agg(
    F.count("*").alias("alunos"),
    F.countDistinct("id_municipio").alias("municipios"),
    F.countDistinct("id_escola").alias("escolas"),
    F.sum(F.when(F.col("presenca") == 1, 1).otherwise(0)).alias("presentes"),
    F.sum(F.when(F.col("alfabetizado") == 1, 1).otherwise(0)).alias("alfabetizados"),
).orderBy("ano").show()

print("\nDistribuição por rede:")
df_check.groupBy("rede").count().orderBy("rede").show()

print("\nAmostra:")
df_check.select(
    "ano", "id_municipio", "id_escola", "id_aluno",
    "rede", "presenca", "alfabetizado", "proficiencia"
).orderBy("ano", "id_municipio").show(10, truncate=False)